## Data Preprocessing / ETL

### Importing Libraries

In [2]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully
Pandas version: 2.1.4
NumPy version: 1.26.2


### Loading Dataset

In [3]:
raw_path = "../data/raw/Sample - Superstore.csv"

df_raw = pd.read_csv(raw_path, encoding='latin-1')

print(f"Dataset loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
print("\nColumn names:")
print(df_raw.columns.tolist())

Dataset loaded: 9994 rows × 21 columns

Column names:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


### Dataset inspection

In [5]:
print("------ First 5 rows ------")
display(df_raw.head())

print("\n------ Data Types ------")
print(df_raw.dtypes)

print("\n------ Basic Shape ------")
print(f"Rows: {df_raw.shape[0]}, Columns: {df_raw.shape[1]}")

------ First 5 rows ------


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164



------ Data Types ------
Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code        int64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

------ Basic Shape ------
Rows: 9994, Columns: 21


### Handling Missing values of the Dataset

In [6]:
print("------ Missing Values ------")
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct.round(2)
})

# Show only columns that have missing values
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print("No missing values found in the dataset!")
else:
    display(missing_df)

------ Missing Values ------
No missing values found in the dataset!


### Handling Duplicates

In [10]:
print("----- Duplicate Rows -----")
dup_count = df_raw.duplicated().sum()
print(f"Total duplicate rows: {dup_count}")

if dup_count > 0:
    print("\nDuplicate rows:")
    display(df_raw[df_raw.duplicated()])
else:
    print("No duplicate rows found")

----- Duplicate Rows -----
Total duplicate rows: 0
No duplicate rows found


### Data Type Conversion

In [11]:
df = df_raw.copy()

# Handling date
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%m/%d/%Y')

# Verify
print("Date columns converted:")
print(f"  Order Date dtype: {df['Order Date'].dtype}")
print(f"  Ship Date dtype:  {df['Ship Date'].dtype}")

Date columns converted:
  Order Date dtype: datetime64[ns]
  Ship Date dtype:  datetime64[ns]


### Feature Engineering

In [12]:
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Quarter'] = df['Order Date'].dt.quarter
df['Day_of_Week'] = df['Order Date'].dt.day_name()

# Shipping duration (days between order and ship)
df['Shipping_Days'] = (df['Ship Date'] - df['Order Date']).dt.days

# Profit Margin (%)
df['Profit_Margin'] = (df['Profit'] / df['Sales']) * 100

# Revenue category
df['Revenue_Category'] = pd.cut(
    df['Sales'],
    bins=[0, 100, 500, 1000, 5000, 100000],
    labels=['Very Low', 'Low', 'Medium', 'High', 'Very High']
)

print("------- New features ------")
print(["Year", "Month", "Quarter", "Day_of_Week", 
       "Shipping_Days", "Profit_Margin", "Revenue_Category"])

display(df[['Order Date', 'Ship Date', 'Shipping_Days', 
            'Profit', 'Sales', 'Profit_Margin', 'Revenue_Category']].head())

------- New features ------
['Year', 'Month', 'Quarter', 'Day_of_Week', 'Shipping_Days', 'Profit_Margin', 'Revenue_Category']


,Order Date,Ship Date,Shipping_Days,Profit,Sales,Profit_Margin,Revenue_Category
0,2016-11-08,2016-11-11,3,41.9136,261.9600,16.00,Low
1,2016-11-08,2016-11-11,3,219.5820,731.9400,30.00,Medium
2,2016-06-12,2016-06-16,4,6.8714,14.6200,47.00,Very Low
3,2015-10-11,2015-10-18,7,-383.0310,957.5775,-40.00,Medium
4,2015-10-11,2015-10-18,7,2.5164,22.3680,11.25,Very Low


### Handling Outliers


```
This outlier handling part only be done for clustering
```

In [13]:
def remove_outliers_iqr(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    before = len(dataframe)
    df_clean = dataframe[(dataframe[column] >= lower) & 
                          (dataframe[column] <= upper)]
    after = len(df_clean)
    print(f"  '{column}': Removed {before - after} outliers | Range: [{lower:.2f}, {upper:.2f}]")
    return df_clean

In [14]:
print("------ Outlier Removal Report ------")

df_clean = df.copy()
for col in ['Sales', 'Profit', 'Discount', 'Shipping_Days']:
    df_clean = remove_outliers_iqr(df_clean, col)

print(f"\nRows before: {len(df)} → After: {len(df_clean)}")
print(f"Rows removed: {len(df) - len(df_clean)}")

------ Outlier Removal Report ------
  'Sales': Removed 1167 outliers | Range: [-271.71, 498.93]
  'Profit': Removed 1435 outliers | Range: [-27.75, 50.78]
  'Discount': Removed 613 outliers | Range: [-0.30, 0.50]
  'Shipping_Days': Removed 0 outliers | Range: [0.00, 8.00]

Rows before: 9994 → After: 6779
Rows removed: 3215


### Column Renaming

In [15]:
df_clean.columns = [c.replace(' ', '_').replace('-', '_') for c in df_clean.columns]

print("Columns renamed:")
print(df_clean.columns.tolist())

Columns renamed:
['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Year', 'Month', 'Quarter', 'Day_of_Week', 'Shipping_Days', 'Profit_Margin', 'Revenue_Category']


### Saving Dataset

In [16]:
output_path = "../data/processed/superstore_clean.csv"
df_clean.to_csv(output_path, index=False)

print(f" Cleaned dataset saved to: {output_path}")
print(f"   Final shape: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")
print(f"\nFinal column list:")
for col in df_clean.columns:
    print(f"  - {col}")

 Cleaned dataset saved to: ../data/processed/superstore_clean.csv
   Final shape: 6779 rows × 28 columns

Final column list:
  - Row_ID
  - Order_ID
  - Order_Date
  - Ship_Date
  - Ship_Mode
  - Customer_ID
  - Customer_Name
  - Segment
  - Country
  - City
  - State
  - Postal_Code
  - Region
  - Product_ID
  - Category
  - Sub_Category
  - Product_Name
  - Sales
  - Quantity
  - Discount
  - Profit
  - Year
  - Month
  - Quarter
  - Day_of_Week
  - Shipping_Days
  - Profit_Margin
  - Revenue_Category


### ETL Summary

In [17]:
print("           ETL PIPELINE SUMMARY REPORT")
print("-" * 60)
print(f"\nEXTRACT")
print(f"   Source  : Sample Superstore CSV (Kaggle)")
print(f"   Raw rows: {df_raw.shape[0]}")
print(f"   Columns : {df_raw.shape[1]}")

print(f"\nTRANSFORM")
print(f"   - Converted Order Date & Ship Date to datetime")
print(f"   - Created 6 new features (Year, Month, Quarter, etc.)")
print(f"   - Removed outliers using IQR method")
print(f"   - Renamed columns to snake_case")
print(f"   - No missing values found (data is clean)")

print(f"\nLOAD")
print(f"   Destination  : ../data/processed/superstore_clean.csv")
print(f"   Final rows   : {df_clean.shape[0]}")
print(f"   Final columns: {df_clean.shape[1]}")

           ETL PIPELINE SUMMARY REPORT
------------------------------------------------------------

EXTRACT
   Source  : Sample Superstore CSV (Kaggle)
   Raw rows: 9994
   Columns : 21

TRANSFORM
   - Converted Order Date & Ship Date to datetime
   - Created 6 new features (Year, Month, Quarter, etc.)
   - Removed outliers using IQR method
   - Renamed columns to snake_case
   - No missing values found (data is clean)

LOAD
   Destination  : ../data/processed/superstore_clean.csv
   Final rows   : 6779
   Final columns: 28
